# Green Computing Project, Academic Year 2025/26

Student: Vavassori Alessandra

Student ID: 942819

Student e-mail adress: a.vavassori11@campus.unimib.it

**Introduction**

For the development of this project, it was chosen to apply Random Forest algorithm on all five datasets. For each dataset, several pre-processing operations were performed, such as checking missing values, analyzing the target distribution and making some adjustments in case of unusual variable values.

After the pre-processing step, it was implemented a "standard" version of the code on each dataset in order to evaluate the result of Matthews Correlation Coefficient (MCC), energy consumption, execution time (in seconds) and CO2 emissions.

Subsequently, a "green" version of the code was developed by applying specific strategies to reduce all the consumption values while maintaining a good performance.



In [ ]:
#importing libraries

!pip install codecarbon
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, confusion_matrix
from sklearn.model_selection import train_test_split

from codecarbon import EmissionsTracker

# First dataset: Neuroblastoma

In [ ]:
df_neuroblastoma = pd.read_csv("/content/data_neuroblastoma.csv")
df_neuroblastoma

,age,sex,site,stage,risk,time_months,autologous_stem_cell_transplantation,radiation,degree_of_differentiation,UH_or_FH,MYCN_status,surgical_methods,outcome
0,2,0,0,1,1,7.0,0,0,0,0,1,1,0
1,2,0,0,1,1,19.0,0,1,0,0,1,1,1
2,1,0,0,1,1,25.0,0,1,1,0,1,0,1
3,1,0,0,1,1,8.0,0,0,1,0,1,1,0
4,1,0,0,1,1,47.0,1,1,1,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,0,0,0,0,0,36.0,0,0,0,0,1,0,0
165,1,0,0,0,0,18.0,0,0,1,1,1,1,1
166,1,1,1,0,0,38.0,0,0,2,1,1,0,0
167,1,1,1,0,0,37.0,0,0,1,1,1,0,0


In [ ]:
#Pre-processing

#number of rows and columns
print(f"Shape dataset:\n {df_neuroblastoma.shape}")

#variable types
print(f"Data types:\n {df_neuroblastoma.dtypes}")

#summary statistics
print(f"Summary statistics:\n {df_neuroblastoma.describe()}")

#distribution of the target variable
print(f"Distribution of the target:\n {df_neuroblastoma['outcome'].value_counts()}")

#missing values
print(f"Missing values per column:\n {df_neuroblastoma.isnull().sum()}")

#one-hot encoding for the categorical variable 'site'
df_neuroblastoma = pd.get_dummies(df_neuroblastoma, columns=['site'], dtype=int)
df_neuroblastoma

Shape dataset:
 (169, 13)
Data types:
 age                                       int64
sex                                       int64
site                                      int64
stage                                     int64
risk                                      int64
time_months                             float64
autologous_stem_cell_transplantation      int64
radiation                                 int64
degree_of_differentiation                 int64
UH_or_FH                                  int64
MYCN_status                               int64
surgical_methods                          int64
outcome                                   int64
dtype: object
Summary statistics:
               age         sex        site       stage        risk  \
count  169.000000  169.000000  169.000000  169.000000  169.000000   
mean     1.124260    0.349112    0.213018    0.686391    0.810651   
std      0.628635    0.478106    0.525149    0.465338    0.392950   
min      0.000000    0.000

,age,sex,stage,risk,time_months,autologous_stem_cell_transplantation,radiation,degree_of_differentiation,UH_or_FH,MYCN_status,surgical_methods,outcome,site_0,site_1,site_2
0,2,0,1,1,7.0,0,0,0,0,1,1,0,1,0,0
1,2,0,1,1,19.0,0,1,0,0,1,1,1,1,0,0
2,1,0,1,1,25.0,0,1,1,0,1,0,1,1,0,0
3,1,0,1,1,8.0,0,0,1,0,1,1,0,1,0,0
4,1,0,1,1,47.0,1,1,1,0,1,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,0,0,0,0,36.0,0,0,0,0,1,0,0,1,0,0
165,1,0,0,0,18.0,0,0,1,1,1,1,1,1,0,0
166,1,1,0,0,38.0,0,0,2,1,1,0,0,0,1,0
167,1,1,0,0,37.0,0,0,1,1,1,0,0,0,1,0


In [ ]:
#Standard code

X = df_neuroblastoma.drop(columns=['outcome'])
y = df_neuroblastoma['outcome']

np.random.seed(42)

mcc_vector = []

tracker_standard = EmissionsTracker(
    log_level="error",
    save_to_file=False
)

tracker_standard.start()
start_time_standard = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Random Forest
    rf_standard = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight='balanced',
        random_state=42
    )

    rf_standard.fit(
        X_train,
        y_train
    )

    y_pred_standard = rf_standard.predict(X_test)

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_standard
    )

    mcc_vector.append(mcc)

end_time_standard = time.time()

execution_time_standard = (end_time_standard - start_time_standard)

emissions_standard = tracker_standard.stop() * 1000

energy_standard_wh = (tracker_standard._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.20f}")
print(f"Median MCC: {np.median(mcc_vector):.20f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.20f}")
print(f"Minimum MCC: {np.min(mcc_vector):.20f}")
print(f"Maximum MCC: {np.max(mcc_vector):.29f}")
print(f"Energy: {energy_standard_wh:.20f} Wh")
print(f"CO2 emissions: {emissions_standard:.20f} CO2 eq grams")
print(f"Execution time: {execution_time_standard:.20f} seconds")

The final results are:
Mean MCC: 0.44417242142400198412
Median MCC: 0.44565217391304345895
Standard deviation MCC: 0.10814807511483337521
Minimum MCC: 0.15521549593647762655
Maximum MCC: 0.72481421078251084821886252030
Energy: 0.06633140248406961537 Wh
CO2 emissions: 0.01893786337351972826 CO2 eq grams
Execution time: 17.66194319725036621094 seconds


In [ ]:
#Green code

X = df_neuroblastoma.drop(columns=['outcome'])
y = df_neuroblastoma['outcome']

np.random.seed(42)

mcc_vector = []

tracker_green = EmissionsTracker(
    log_level="error",
    save_to_file=False
)

tracker_green.start()

start_time_green = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Random Forest
    rf_green = RandomForestClassifier(
        n_estimators=20,
        max_depth=5,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42
    )

    rf_green.fit(
        X_train,
        y_train
    )

    y_pred_green = rf_green.predict(
        X_test)

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_green
    )

    mcc_vector.append(mcc)

end_time_green = time.time()

execution_time_green = (
    end_time_green - start_time_green)

emissions_green = tracker_green.stop() * 1000

energy_green_wh = (tracker_green._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.20f}")
print(f"Median MCC: {np.median(mcc_vector):.20f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.20f}")
print(f"Minimum MCC: {np.min(mcc_vector):.20f}")
print(f"Maximum MCC: {np.max(mcc_vector):.20f}")
print(f"\nEnergy: {energy_green_wh:.20f} Wh")
print(f"CO2 emissions: {emissions_green:.20f} CO2 eq grams")
print(f"Execution time: {execution_time_green:.20f} seconds")

The final results are:
Mean MCC: 0.51370486388612301987
Median MCC: 0.52279451819237399146
Standard deviation MCC: 0.10597273695077616407
Minimum MCC: 0.24493247972574661020
Maximum MCC: 0.72481421078251084822

Energy: 0.01667962261167760946 Wh
CO2 emissions: 0.00476209460847270422 CO2 eq grams
Execution time: 4.25146222114562988281 seconds


Exclusively on the first dataset it was applied another green strategy which consisted in feature selection based on the ranking of important features and selecting only the first ten (k=10).

However, it was noticed that the results collected through this strategy for energy consumptions, emissions and execution time were actually worse compared to the ones obtained with the strategy based solely on hyperparameter modification.
Consequently, this strategy was discarded and for the remaining datasets it was applied only the modification of hyperparameters.

Further details and explanation are provided in the section of the report entitled “4.2 Evaluation of an additional strategy: Feature Selection".

In [ ]:
#Optional strategy applied only on the first dataset: Feature Selection based on importance ranking

X = df_neuroblastoma.drop(columns=['outcome'])
y = df_neuroblastoma['outcome']

np.random.seed(42)

mcc_vector = []

#number of features kept
k = 10

tracker_green = EmissionsTracker(
    log_level="error",
    save_to_file=False
)

tracker_green.start()

start_time_green = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Feature selection only on the training set
    rf_selector = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
    rf_selector.fit(X_train, y_train)

    #ranking of features according to importance
    feature_importances = rf_selector.feature_importances_

    #select top 10 features
    top_k_indices = np.argsort(feature_importances)[::-1][:k]
    selected_features = X_train.columns[top_k_indices]

    X_train_selected = X_train[selected_features]
    X_test_selected = X_test[selected_features]

    #Random Forest
    rf_green = RandomForestClassifier(
        n_estimators=20,
        max_depth=5,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42
    )

    rf_green.fit(
        X_train_selected,
        y_train)

    y_pred_green = rf_green.predict(
        X_test_selected)

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_green
    )

    mcc_vector.append(mcc)

end_time_green = time.time()

execution_time_green = (end_time_green - start_time_green)

emissions_green = tracker_green.stop() * 1000

energy_green_wh = (tracker_green._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.20f}")
print(f"Median MCC: {np.median(mcc_vector):.20f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.20f}")
print(f"Minimum MCC: {np.min(mcc_vector):.20f}")
print(f"Maximum MCC: {np.max(mcc_vector):.20f}")
print(f"\nEnergy: {energy_green_wh:.20f} Wh")
print(f"CO2 emissions: {emissions_green:.20f} CO2 eq grams")
print(f"Execution time: {execution_time_green:.20f} seconds")

The final results are:
Mean MCC: 0.53147481778833993715
Median MCC: 0.52712951122173978824
Standard deviation MCC: 0.10538781186965831549
Minimum MCC: 0.24371482196617177962
Maximum MCC: 0.76242236024844722841

Energy: 0.08088701595892311413 Wh
CO2 emissions: 0.02309354543332149332 CO2 eq grams
Execution time: 21.62535667419433593750 seconds


# Second dataset: Pediatric brain tumor

In [ ]:
df_pediatric = pd.read_csv("/content/dataset_Belgrade2021_pediatric_brain_tumor.csv")

In [ ]:
#Pre-processing

#number of rows and columns
print(f"Shape dataset:\n {df_pediatric.shape}")

#variable types
print(f"Data types:\n {df_pediatric.dtypes}")

#summary statistics
print(f"Summary statistics:\n {df_pediatric.describe()}")

#distribution of target variable
print(f"Distribuzione target:\n {df_pediatric['Status OS'].value_counts()}")

#missing values
print(f"Missing values per column:\n {df_pediatric.isnull().sum()}")

#one-hot encoding for the categorical variable 'operation type'
df_pediatric = pd.get_dummies(df_pediatric, columns=['Operation type'], dtype=int)
df_pediatric

Shape dataset:
 (173, 31)
Data types:
 Sex                                                int64
Age                                                int64
Duration of sympthoms before diagnosis (days)    float64
Increased ICP                                      int64
Epileptic seizures                                 int64
Neurological deficit                               int64
Hormonal abnormalities                             int64
Operation type                                     int64
Localisation                                     float64
Embryonal tumors                                   int64
HGG                                                int64
LGG                                                int64
Craniopharyngeoma                                  int64
GCT / NGCT                                         int64
Ependymoma                                         int64
Glioneural Tu                                      int64
Pineal Tu                                        

,Sex,Age,Duration of sympthoms before diagnosis (days),Increased ICP,Epileptic seizures,Neurological deficit,Hormonal abnormalities,Localisation,Embryonal tumors,HGG,...,CSI dose,Total dose,Concomitant HT,Adjuvant HT,OS,Status OS,Operation type_1,Operation type_2,Operation type_3,Operation type_4
0,1,1996,36.0,1,2,1,2,2.0,2,2,...,30.6,54.00,2.0,2,147.4,2,0,1,0,0
1,1,1996,21.0,1,2,1,2,2.0,1,2,...,35.2,55.25,2.0,2,19.0,1,0,0,0,1
2,2,2001,30.0,2,2,1,2,2.0,2,2,...,30.6,50.32,2.0,2,6.0,1,0,0,1,0
3,2,2001,365.0,2,2,1,1,1.0,2,2,...,30.6,54.00,2.0,2,147.4,2,0,0,1,0
4,1,1993,33.0,1,2,2,2,1.0,2,2,...,30.6,55.80,2.0,2,146.0,2,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,1,2006,14.0,1,1,2,2,1.0,2,2,...,34.2,55.80,2.0,1,24.0,1,0,0,1,0
169,1,2009,20.0,1,2,2,2,1.0,2,1,...,30.6,55.80,1.6,1,17.0,1,0,0,0,1
170,2,2008,14.0,1,2,1,2,2.0,1,2,...,34.2,54.00,2.0,1,14.0,1,0,0,0,1
171,1,2011,11.6,2,2,1,2,2.0,1,2,...,30.6,50.32,2.0,2,6.0,1,0,0,1,0


In [ ]:
#Standard code

X = df_pediatric.drop(columns=['Status OS'])
y = df_pediatric['Status OS']

np.random.seed(42)

mcc_vector = []

tracker_standard = EmissionsTracker(
    log_level="error",
    save_to_file=False
)

tracker_standard.start()

start_time_standard = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Random Forest
    rf_standard = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight='balanced',
        random_state=42
    )

    rf_standard.fit(
        X_train,
        y_train
    )

    y_pred_standard = rf_standard.predict(
        X_test
    )

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_standard
    )

    mcc_vector.append(mcc)

end_time_standard = time.time()

execution_time_standard = (end_time_standard - start_time_standard)

emissions_standard = tracker_standard.stop() * 1000

energy_standard_wh = (tracker_standard._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.20f}")
print(f"Median MCC: {np.median(mcc_vector):.20f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.20f}")
print(f"Minimum MCC: {np.min(mcc_vector):.20f}")
print(f"Maximum MCC: {np.max(mcc_vector):.20f}")
print(f"\nEnergy: {energy_standard_wh:.20f} Wh")
print(f"CO2 emissions: {emissions_standard:.20f} CO2 eq grams")
print(f"Execution time: {execution_time_standard:.20f} seconds")

The final results are:
Mean MCC: 0.77821204511927877245
Median MCC: 0.77462068290289820549
Standard deviation MCC: 0.07959529593802183722
Minimum MCC: 0.41903519109970349676
Maximum MCC: 0.92496246170077389692

Energy: 0.13032101473726587226 Wh
CO2 emissions: 0.03720713688189415669 CO2 eq grams
Execution time: 28.56003522872924804688 seconds


In [ ]:
#Green code

X = df_pediatric.drop(columns=['Status OS'])
y = df_pediatric['Status OS']

np.random.seed(42)

mcc_vector = []

tracker_green = EmissionsTracker(
    log_level="error",
    save_to_file=False
)

tracker_green.start()

start_time_green = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Random Forest
    rf_green = RandomForestClassifier(
        n_estimators=20,
        max_depth=5,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42
    )

    rf_green.fit(
        X_train,
        y_train
    )

    y_pred_green = rf_green.predict(
        X_test)

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_green
    )

    mcc_vector.append(mcc)

end_time_green = time.time()

execution_time_green = (end_time_green - start_time_green)

emissions_green = tracker_green.stop() * 1000

energy_green_wh = (tracker_green._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.20f}")
print(f"Median MCC: {np.median(mcc_vector):.20f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.20f}")
print(f"Minimum MCC: {np.min(mcc_vector):.20f}")
print(f"Maximum MCC: {np.max(mcc_vector):.20f}")
print(f"\nEnergy: {energy_green_wh:.20f} Wh")
print(f"CO2 emissions: {emissions_green:.20f} CO2 eq grams")
print(f"Execution time: {execution_time_green:.20f} seconds")

The final results are:
Mean MCC: 0.79982218333099242003
Median MCC: 0.80657769207694995917
Standard deviation MCC: 0.06722511677993665835
Minimum MCC: 0.60258274378375875635
Maximum MCC: 0.92582009977255153022

Energy: 0.01510871247315800393 Wh
CO2 emissions: 0.00431359389144797458 CO2 eq grams
Execution time: 3.63488578796386718750 seconds


# Third Dataset: Colorectal cancer

In [ ]:
df_colcancer=pd.read_csv("/content/dataset_Taipei2018_colorectal_cancer.csv")
df_colcancer

,bian_ma,age,gender,asa,asa3,dm,cad,hf,cva,ckd,...,signet_ring,lymphovascularinvasion,perineural,ct,rt,nactrt,interval,progress,interval_r,death
0,2,52,1,3,1,0,0,0,0,0,...,0,1,0,1,0,0,11.761807,1,9.856263,1
1,21,85,2,2,0,0,1,0,0,0,...,0,0,0,1,0,0,8.377823,1,7.063655,1
2,22,45,2,2,0,0,0,0,0,0,...,0,1,0,1,0,0,4.763860,1,2.792608,1
3,23,57,2,2,0,0,0,0,0,0,...,0,1,0,1,0,0,2.595483,1,1.774127,1
4,29,43,1,2,0,1,0,0,0,0,...,0,0,0,1,0,0,16.164271,1,7.096509,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,5158,48,1,3,1,0,0,0,0,0,...,0,0,0,1,0,0,18.398357,0,18.398357,0
995,5162,86,1,3,1,0,0,0,0,0,...,0,0,0,0,0,1,17.478439,1,8.410678,0
996,5168,49,2,2,0,0,0,0,0,0,...,0,1,1,0,0,0,0.131417,0,0.131417,0
997,5171,76,2,2,0,1,0,0,0,0,...,0,1,1,1,0,0,20.008214,1,16.558522,0


In [ ]:
#variables inspection
df_colcancer.columns

df_colcancer["asa"].value_counts()
df_colcancer["cea"].value_counts()
df_colcancer["log_cea"].value_counts()

#dropping the variable
df_colcancer.drop('cea', axis=1, inplace=True)

#next variables give values: 0, 1, 9
print(df_colcancer["cva"].value_counts())
print(df_colcancer["cell_diff"].value_counts())
print(df_colcancer["mucin_type"].value_counts())
print(df_colcancer["signet_ring"].value_counts())
print(df_colcancer["lymphovascularinvasion"].value_counts())
print(df_colcancer["perineural"].value_counts())


cva
0    942
1     57
Name: count, dtype: int64
cell_diff
1    813
2    121
9     65
Name: count, dtype: int64
mucin_type
0    869
1     73
9     57
Name: count, dtype: int64
signet_ring
0    900
9     57
1     42
Name: count, dtype: int64
lymphovascularinvasion
1    485
0    460
9     54
Name: count, dtype: int64
perineural
0    734
1    208
9     57
Name: count, dtype: int64


In [ ]:
df_clean = df_colcancer[
    (df_colcancer['mucin_type'] != 9) &
    (df_colcancer['cell_diff'] != 9) &
    (df_colcancer['signet_ring'] != 9)  &
    (df_colcancer['lymphovascularinvasion'] != 9) &
    (df_colcancer['perineural'] != 9)
]
df_clean

,bian_ma,age,gender,asa,asa3,dm,cad,hf,cva,ckd,...,signet_ring,lymphovascularinvasion,perineural,ct,rt,nactrt,interval,progress,interval_r,death
0,2,52,1,3,1,0,0,0,0,0,...,0,1,0,1,0,0,11.761807,1,9.856263,1
1,21,85,2,2,0,0,1,0,0,0,...,0,0,0,1,0,0,8.377823,1,7.063655,1
2,22,45,2,2,0,0,0,0,0,0,...,0,1,0,1,0,0,4.763860,1,2.792608,1
3,23,57,2,2,0,0,0,0,0,0,...,0,1,0,1,0,0,2.595483,1,1.774127,1
4,29,43,1,2,0,1,0,0,0,0,...,0,0,0,1,0,0,16.164271,1,7.096509,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,5158,48,1,3,1,0,0,0,0,0,...,0,0,0,1,0,0,18.398357,0,18.398357,0
995,5162,86,1,3,1,0,0,0,0,0,...,0,0,0,0,0,1,17.478439,1,8.410678,0
996,5168,49,2,2,0,0,0,0,0,0,...,0,1,1,0,0,0,0.131417,0,0.131417,0
997,5171,76,2,2,0,1,0,0,0,0,...,0,1,1,1,0,0,20.008214,1,16.558522,0


In [ ]:
#Pre-processing

#number of rows and columns
print(f"Shape dataset:\n {df_clean.shape}")

#variable types
print(f"Data types:\n {df_clean.dtypes}")

#summary statistics
print(f"Summary statistics:\n {df_clean.describe()}")

#distribution of the target variable
print(f"Distribution of the target:\n {df_clean['death'].value_counts()}")

#missing values
print(f"Missing values per column:\n {df_clean.isnull().sum()}")

Shape dataset:
 (931, 31)
Data types:
 bian_ma                     int64
age                         int64
gender                      int64
asa                         int64
asa3                        int64
dm                          int64
cad                         int64
hf                          int64
cva                         int64
ckd                         int64
log_cea                   float64
laparoscopic                int64
tumor_loc                   int64
ea                          int64
anes_time                   int64
log2at                    float64
rbc                         int64
ajcc                        int64
liver_only                  int64
cell_diff                   int64
mucin_type                  int64
signet_ring                 int64
lymphovascularinvasion      int64
perineural                  int64
ct                          int64
rt                          int64
nactrt                      int64
interval                  float64
progress 

In [ ]:
#Standard code

X = df_clean.drop(columns=['death'])
y = df_clean['death']

np.random.seed(42)

mcc_vector = []

tracker_standard = EmissionsTracker(
    log_level="error",
    save_to_file=False
)

tracker_standard.start()

start_time_standard = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Random Forest
    rf_standard = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight='balanced',
        random_state=42
    )

    rf_standard.fit(
        X_train,
        y_train
    )

    y_pred_standard = rf_standard.predict(
        X_test)

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_standard
    )

    mcc_vector.append(mcc)

end_time_standard = time.time()

execution_time_standard = (end_time_standard - start_time_standard)

emissions_standard = tracker_standard.stop() * 1000

energy_standard_wh = (tracker_standard._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.20f}")
print(f"Median MCC: {np.median(mcc_vector):.20f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.20f}")
print(f"Minimum MCC: {np.min(mcc_vector):.20f}")
print(f"Maximum MCC: {np.max(mcc_vector):.20f}")
print(f"\nEnergy: {energy_standard_wh:.20f} Wh")
print(f"CO2 emissions: {emissions_standard:.20f}  CO2 eq grams")
print(f"Execution time: {execution_time_standard:.20f} seconds")

The final results are:
Mean MCC: 0.22116466611985494128
Median MCC: 0.22517502749427431574
Standard deviation MCC: 0.05099781197213792278
Minimum MCC: 0.05185485539500656049
Maximum MCC: 0.33296064306819689316

Energy: 0.19842633005044835182 Wh
CO2 emissions: 0.05665145899948071523  CO2 eq grams
Execution time: 44.09179878234863281250 seconds


In [ ]:
#Green code

X = df_clean.drop(columns=['death'])
y = df_clean['death']

np.random.seed(42)

mcc_vector = []

tracker_green = EmissionsTracker(
    log_level="error",
    save_to_file=False)

tracker_green.start()

start_time_green = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Random Forest
    rf_green = RandomForestClassifier(
        n_estimators=20,
        max_depth=5,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42
    )

    rf_green.fit(
        X_train,
        y_train
    )

    y_pred_green = rf_green.predict(
        X_test)

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_green
    )

    mcc_vector.append(mcc)

end_time_green = time.time()

execution_time_green = (end_time_green - start_time_green)

emissions_green = tracker_green.stop() * 1000

energy_green_wh = (tracker_green._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.20f}")
print(f"Median MCC: {np.median(mcc_vector):.20f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.20f}")
print(f"Minimum MCC: {np.min(mcc_vector):.20f}")
print(f"Maximum MCC: {np.max(mcc_vector):.20f}")
print(f"\nEnergy: {energy_green_wh:.20f} Wh")
print(f"CO2 emissions: {emissions_green:.20f} CO2 eq grams")
print(f"Execution time: {execution_time_green:.20f} seconds")

The final results are:
Mean MCC: 0.25237408305332154068
Median MCC: 0.24999284143240160794
Standard deviation MCC: 0.05626590644955356546
Minimum MCC: 0.07232717276712437304
Maximum MCC: 0.38845923161818174396

Energy: 0.01937262758913455007 Wh
CO2 emissions: 0.00553095759670118974 CO2 eq grams
Execution time: 5.07690572738647460938 seconds


# Fourth Dataset: Depression heart failure

In [ ]:
df_dep_heart_failure = pd.read_csv("/content/depression_heart_failure.csv")
df_dep_heart_failure = df_dep_heart_failure.rename(columns={"Death (1=Yes, 0=No)": "Death"})
df_dep_heart_failure

,Age (years),"Male (1=Yes, 0=No)",PHQ-9,Systolic BP (mm Hg),Estimated glomerular filtration rate,Ejection fraction (%),Serum sodium (mmol/l),Blood urea nitrogen (mg/dl),"Etiology HF(1=Yes, 0=No)",Prior diabetes mellitus,"Elevated level of BNP/NT-BNP (1=Yes, 0=No)",Time from HF to Death (days),Time from HF to hospitalization (days),"Hospitalized (1=Yes, 0=No)",Death
0,93,0,1,145.0,46.352457,60.50,137,26,1,1,1,679,134,1,1
1,99,1,8,115.0,32.827851,65.00,140,75,1,0,1,74,74,0,1
2,83,1,4,121.0,57.821591,65.00,141,28,0,0,1,730,224,1,0
3,86,1,4,113.0,63.469884,32.50,141,47,0,1,1,730,5,1,0
4,84,1,3,142.0,57.681188,58.00,137,32,0,1,1,730,730,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,50,1,3,143.0,79.094655,49.50,137,27,1,1,1,730,156,1,0
421,73,1,11,155.0,29.486806,58.00,139,42,0,0,1,730,730,0,0
422,63,1,17,135.0,113.901326,27.40,130,17,0,1,1,338,338,0,1
423,48,1,4,172.0,58.919011,16.83,140,34,0,0,1,730,730,0,0


In [ ]:
#Pre-processing

#number of rows and columns
print(f"Shape dataset:\n {df_dep_heart_failure.shape}")

#variable types
print(f"Data types:\n {df_dep_heart_failure.dtypes}")

#summary statistics
print(f"Summary statistics:\n  {df_dep_heart_failure.describe()}")

#distribution of target variable
print(f"Distribution target:\n {df_dep_heart_failure["Death"].value_counts()}")

#missing values
print(f"Missing values per column:\n {df_dep_heart_failure.isnull().sum()}")

Shape dataset:
 (425, 15)
Data types:
 Age (years)                                     int64
Male (1=Yes, 0=No)                              int64
PHQ-9                                           int64
Systolic BP (mm Hg)                           float64
Estimated glomerular filtration rate          float64
Ejection fraction (%)                         float64
Serum sodium (mmol/l)                           int64
Blood urea nitrogen (mg/dl)                     int64
Etiology HF(1=Yes, 0=No)                        int64
Prior diabetes mellitus                         int64
Elevated level of BNP/NT-BNP (1=Yes, 0=No)      int64
Time from HF to Death (days)                    int64
Time from HF to hospitalization (days)          int64
Hospitalized (1=Yes, 0=No)                      int64
Death                                           int64
dtype: object
Summary statistics:
         Age (years)  Male (1=Yes, 0=No)       PHQ-9  Systolic BP (mm Hg)  \
count   425.000000          425.000000  

In [ ]:
#correlation coefficient
df_dep_heart_failure.corr()["Death"].sort_values()

,Death
Time from HF to Death (days),-0.858882
Time from HF to hospitalization (days),-0.304771
Estimated glomerular filtration rate,-0.290426
Systolic BP (mm Hg),-0.110014
"Male (1=Yes, 0=No)",-0.045856
Serum sodium (mmol/l),-0.019627
Ejection fraction (%),-0.002139
Prior diabetes mellitus,0.057589
"Etiology HF(1=Yes, 0=No)",0.110744
"Elevated level of BNP/NT-BNP (1=Yes, 0=No)",0.128198


In [ ]:
#Standard code

X = df_dep_heart_failure.drop(columns=["Death", "Time from HF to Death (days)", "Time from HF to hospitalization (days)"])
y = df_dep_heart_failure["Death"]

np.random.seed(42)

mcc_vector = []

tracker_standard = EmissionsTracker(
    log_level="error",
    save_to_file=False)

tracker_standard.start()

start_time_standard = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Random Forest
    rf_standard = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight='balanced',
        random_state=42
    )

    rf_standard.fit(
        X_train,
        y_train
    )

    y_pred_standard = rf_standard.predict(
        X_test)

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_standard
    )
    mcc_vector.append(mcc)

end_time_standard = time.time()

execution_time_standard = (end_time_standard - start_time_standard)

emissions_standard = tracker_standard.stop() * 1000

energy_standard_wh = (tracker_standard._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.20f}")
print(f"Median MCC: {np.median(mcc_vector):.20f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.20f}")
print(f"Minimum MCC: {np.min(mcc_vector):.20f}")
print(f"Maximum MCC: {np.max(mcc_vector):.20f}")
print(f"\nEnergy: {energy_standard_wh:.20f} Wh")
print(f"CO2 emissions: {emissions_standard:.20f}  CO2 eq grams")
print(f"Execution time: {execution_time_standard:.20f} seconds")

The final results are:
Mean MCC: 0.25341165174562874096
Median MCC: 0.25126452936094562096
Standard deviation MCC: 0.08968759564987312249
Minimum MCC: 0.02915026191924190158
Maximum MCC: 0.45743029704172155636

Energy: 0.17898474893835511490 Wh
CO2 emissions: 0.05110081491420832150  CO2 eq grams
Execution time: 30.20840549468994140625 seconds


In [ ]:
#Green code

X = df_dep_heart_failure.drop(columns=["Death", "Time from HF to Death (days)", "Time from HF to hospitalization (days)"])
y = df_dep_heart_failure["Death"]

np.random.seed(42)

mcc_vector = []

tracker_green = EmissionsTracker(
    log_level="error",
    save_to_file=False)

tracker_green.start()

start_time_green = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Random Forest
    rf_green = RandomForestClassifier(
        n_estimators=20,
        max_depth=5,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42
    )

    rf_green.fit(
        X_train,
        y_train)

    y_pred_green = rf_green.predict(
        X_test)

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_green
    )

    mcc_vector.append(mcc)

end_time_green = time.time()

execution_time_green = (end_time_green - start_time_green)

emissions_green = tracker_green.stop() * 1000

energy_green_wh = (tracker_green._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.4f}")
print(f"Median MCC: {np.median(mcc_vector):.4f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.4f}")
print(f"Minimum MCC: {np.min(mcc_vector):.4f}")
print(f"Maximum MCC: {np.max(mcc_vector):.4f}")
print(f"\nEnergy: {energy_green_wh:.6f} Wh")
print(f"CO2 emissions: {emissions_green:.6f} CO2 eq grams")
print(f"Execution time: {execution_time_green:.4f} seconds")

The final results are:
Mean MCC: 0.3343
Median MCC: 0.3338
Standard deviation MCC: 0.0868
Minimum MCC: 0.1266
Maximum MCC: 0.5482

Energy: 0.016747 Wh
CO2 emissions: 0.004781 CO2 eq grams
Execution time: 4.2695 seconds


# Fifth Dataset: Sepsis

In [ ]:
df_sepsis = pd.read_csv("/content/journal_Text_Sepsis.csv")
df_sepsis

,Age,sex_woman,diagnosis_0EC_1M_2_AC,APACHE II,SOFA,CRP,WBCC,NeuC,LymC,EOC,NLCR,PLTC,MPV,Group,LOS-ICU,Mortality
0,23,0,2,14,0,0.02,10.48,8.66,1.22,40,7.1,176,10.0,0,1,0
1,53,0,2,10,3,9.00,19.16,16.28,0.62,0,26.3,280,10.8,0,1,0
2,83,0,2,8,4,11.91,12.60,9.91,0.72,0,13.8,98,9.2,0,1,0
3,35,0,2,3,0,7.38,7.70,5.97,0.94,20,6.4,254,7.9,0,1,0
4,86,0,2,17,2,4.97,10.64,9.73,0.78,0,12.5,133,11.8,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1252,84,1,1,18,3,0.30,6.71,5.26,0.81,30,6.5,145,9.7,0,6,0
1253,30,1,1,3,0,0.10,7.89,3.43,3.47,30,1.0,268,10.3,0,1,0
1254,62,1,1,19,6,4.50,4.49,3.53,0.31,20,11.4,188,8.6,0,15,0
1255,31,1,1,16,5,2.00,8.40,6.69,0.76,20,8.8,164,9.0,1,4,0


In [ ]:
#Pre-processing

#number of rows and columns
print(f"Shape dataset:\n {df_sepsis.shape}")

#variable types
print(f"Data types:\n {df_sepsis.dtypes}")

#summary statistics
print(f"Summary statistics:\n {df_sepsis.describe()}")

#distribution of target variable
print(f"Distribution of the target:\n {df_sepsis["Mortality"].value_counts()}")

#missing values
print(f"Missing values per column:\n {df_sepsis.isnull().sum()}")

Shape dataset:
 (1257, 16)
Data types:
 Age                        int64
sex_woman                  int64
diagnosis_0EC_1M_2_AC      int64
APACHE II                  int64
SOFA                       int64
CRP                      float64
WBCC                     float64
NeuC                     float64
LymC                     float64
EOC                        int64
NLCR                     float64
PLTC                       int64
MPV                      float64
Group                      int64
LOS-ICU                    int64
Mortality                  int64
dtype: object
Summary statistics:
                Age    sex_woman  diagnosis_0EC_1M_2_AC    APACHE II  \
count  1257.000000  1257.000000            1257.000000  1257.000000   
mean     56.505967     0.408115               0.546539    13.336516   
std      19.235512     0.491680               0.566816     8.209763   
min      18.000000     0.000000               0.000000     0.000000   
25%      41.000000     0.000000           

In [ ]:
#Standard code

X = df_sepsis.drop(columns=['Mortality'])
y = df_sepsis['Mortality']

np.random.seed(42)

mcc_vector = []

tracker_standard = EmissionsTracker(
    log_level="error",
    save_to_file=False
)

tracker_standard.start()

start_time_standard = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Random Forest
    rf_standard = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight='balanced',
        random_state=42
    )

    rf_standard.fit(
        X_train,
        y_train
    )

    y_pred_standard = rf_standard.predict(
        X_test)

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_standard
    )

    mcc_vector.append(mcc)

end_time_standard = time.time()

execution_time_standard = (end_time_standard - start_time_standard)

emissions_standard = tracker_standard.stop() * 1000

energy_standard_wh = (tracker_standard._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.20f}")
print(f"Median MCC: {np.median(mcc_vector):.20f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.20f}")
print(f"Minimum MCC: {np.min(mcc_vector):.20f}")
print(f"Maximum MCC: {np.max(mcc_vector):.20f}")
print(f"\nEnergy: {energy_standard_wh:.20f} Wh")
print(f"CO2 emissions: {emissions_standard:.20f} CO2 eq grams")
print(f"Execution time: {execution_time_standard:.20f} seconds")

The final results are:
Mean MCC: 0.49551891613907378398
Median MCC: 0.49577120947674374607
Standard deviation MCC: 0.06319988792079596351
Minimum MCC: 0.38604410036067665635
Maximum MCC: 0.64129752839129172681

Energy: 0.16180171713010391144 Wh
CO2 emissions: 0.04619499509823719180 CO2 eq grams
Execution time: 37.04455256462097167969 seconds


In [ ]:
#Green code

X = df_sepsis.drop(columns=['Mortality'])
y = df_sepsis['Mortality']

np.random.seed(42)

mcc_vector = []

tracker_green = EmissionsTracker(
    log_level="error",
    save_to_file=False
)

tracker_green.start()

start_time_green = time.time()

for i in range(100):
    #train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        stratify=y,
        random_state=i
    )

    #Random Forest
    rf_green = RandomForestClassifier(
        n_estimators=20,
        max_depth=5,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42
    )

    rf_green.fit(
        X_train,
        y_train
    )

    y_pred_green = rf_green.predict(
        X_test)

    #MCC
    mcc = matthews_corrcoef(
        y_test,
        y_pred_green)

    mcc_vector.append(mcc)

end_time_green = time.time()

execution_time_green = (end_time_green - start_time_green)

emissions_green = tracker_green.stop() * 1000

energy_green_wh = (tracker_green._total_energy.kWh * 1000)

print("The final results are:")

print(f"Mean MCC: {np.mean(mcc_vector):.20f}")
print(f"Median MCC: {np.median(mcc_vector):.20f}")
print(f"Standard deviation MCC: {np.std(mcc_vector):.20f}")
print(f"Minimum MCC: {np.min(mcc_vector):.20f}")
print(f"Maximum MCC: {np.max(mcc_vector):.20f}")
print(f"\nEnergy: {energy_green_wh:.20f} Wh")
print(f"CO2 emissions: {emissions_green:.20f} CO2 eq grams")
print(f"Execution time: {execution_time_green:.20f} seconds")

The final results are:
Mean MCC: 0.58093971236008401515
Median MCC: 0.58617846343864810077
Standard deviation MCC: 0.04893989227278711107
Minimum MCC: 0.45199166799462980881
Maximum MCC: 0.69820413248459489708

Energy: 0.04594324608138774385 Wh
CO2 emissions: 0.01311696850423560955 CO2 eq grams
Execution time: 8.97203373908996582031 seconds
